# Recomendaciones aplicadas

*Cómo cambiaría la proyección 2018 si el negocio aplica las recomendaciones del EDA*

El modelo de proyección solo conoce el calendario y las ventas pasadas, así que no puede recibir una
acción del negocio. Por eso cada recomendación se aplica como un efecto sobre la proyección base:

> **Proyección con recomendaciones = proyección base 2018 (Random Forest) × efecto de cada recomendación**

| Recomendación | De dónde sale el efecto |
|---|---|
| Tope de descuento | Un segundo modelo de machine learning, que predice el margen de cada línea de pedido según su descuento (`model/train_descuentos.py`) |
| Reforzar la temporada alta | Supuesto: un % más de ventas de septiembre a diciembre |
| Impulsar una categoría | Supuesto: un % más de ventas de esa categoría |

El descuento es la única acción del negocio que registran los datos, por eso es la única que se puede
estimar con un modelo. En la app Dash el simulador es interactivo; aquí se muestran los escenarios
principales.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from common import NEG, POS, PROY, REAL, REF, kpis, load_artifact, load_mensual, md, show, style_fig, tabla, usd

# Proyección base 2018 (model/train_model.py) y escenarios del tope de descuento (model/train_descuentos.py).
# Cada escenario resume, por categoría × región, los pedidos de 2017 antes y después de aplicar el tope.
p18 = load_artifact("proyeccion_2018.csv")
esc = load_artifact("descuentos_escenarios.csv").assign(tope=lambda d: (d.tope * 100).round().astype(int))
TOPES = sorted(esc.tope.unique())  # descuentos que existen en los datos: 0, 10, 15, 20, 30 ... 80 %


def efecto_tope(tope, region="Todas", perdida=0.25):
    """Por categoría × región: factor de ventas y margen después del tope, sobre los pedidos de 2017.
    perdida = fracción de las líneas con descuento recortado que ya no se venden (supuesto)."""
    e = esc[esc.tope == tope].set_index(["Category", "Region"])
    aplica = (e.index.get_level_values("Region") == region) if region != "Todas" else np.ones(len(e), bool)
    # Las líneas que el tope no toca quedan igual; las recortadas pagan más y una parte se pierde
    ventas = np.where(aplica, e.ventas_base - e.ventas_cambian_antes + e.ventas_cambian_despues * (1 - perdida),
                      e.ventas_base)
    ganancia = np.where(aplica, e.ganancia_base - e.ganancia_cambian_antes
                        + e.ganancia_cambian_despues * (1 - perdida), e.ganancia_base)
    return pd.DataFrame({"margen_base": e.ganancia_base / e.ventas_base, "factor_ventas": ventas / e.ventas_base,
                         "margen_sim": ganancia / ventas}, index=e.index)


def simular(tope=80, region="Todas", perdida=0.25, temporada=0, categoria=None, impulso=0):
    """Proyección 2018 por categoría × región × mes: base (v_base, g_base) y con cada recomendación en orden.
    tope=80 equivale a no aplicar tope (es el descuento máximo de los datos)."""
    p = p18.join(efecto_tope(tope, region, perdida), on=["Category", "Region"])
    p["v_base"] = p.Proyeccion
    p["g_base"] = p.v_base * p.margen_base          # ganancia base: margen de 2017 de cada serie
    p["v1"] = p.v_base * p.factor_ventas            # 1) tope de descuento: cambia ventas y margen
    p["g1"] = p.v1 * p.margen_sim
    alta = p.Mes.dt.month >= 9                      # 2) temporada alta: + % de septiembre a diciembre
    p["v2"] = p.v1 * np.where(alta, 1 + temporada / 100, 1)
    p["g2"] = p.v2 * p.margen_sim
    p["v3"] = p.v2 * np.where(p.Category == categoria, 1 + impulso / 100, 1)  # 3) impulso de una categoría
    p["g3"] = p.v3 * p.margen_sim
    return p


def ganancia_2017(tope, region="Todas", perdida=0.25):
    """Ganancia que habría tenido 2017 con ese tope."""
    e = esc[esc.tope == tope].set_index(["Category", "Region"])
    ef = efecto_tope(tope, region, perdida)
    return float((e.ventas_base * ef.factor_ventas * ef.margen_sim).sum())

## Cómo se estima el efecto del descuento

Cada fila es una línea de pedido. El modelo predice su margen (ganancia ÷ venta) según el descuento,
la cantidad, la subcategoría, la región y el segmento, con el mismo método del capítulo *Modelos y
resultados*: entrenamiento 2014–2016, prueba 2017, Grid Search con validación cruzada temporal y
elección por validación cruzada.

In [ ]:
r = load_artifact("descuentos_resultados.csv")
s = load_artifact("descuentos_resumen.json")
ref = r.iloc[0]
# Modelos ordenados por su error de validación cruzada, con la referencia al final
t = pd.concat([r.iloc[1:].sort_values("cv_MAE_margen_pts"), r.iloc[[0]]])
display(tabla(t[["modelo", "cv_MAE_margen_pts", "MAE_margen_pts", "desvio_ganancia_pct"]].rename(columns={
    "modelo": "Modelo", "cv_MAE_margen_pts": "Error CV (puntos de margen)", "MAE_margen_pts": "Error en 2017 (puntos)",
    "desvio_ganancia_pct": "Desvío de la ganancia 2017"})).format(
    {"Error CV (puntos de margen)": "{:.2f}", "Error en 2017 (puntos)": "{:.2f}", "Desvío de la ganancia 2017": "{:+.1f} %"},
    na_rep="—"))
md(f"Se eligió **{s['mejor_modelo']}** por validación cruzada. En 2017 se equivoca en "
   f"{s['prueba']['MAE_margen_pts']:.1f} puntos de margen por línea, casi lo mismo que la referencia "
   f"(margen promedio de la subcategoría con ese descuento, {ref.MAE_margen_pts:.1f}): no la supera en precisión. "
   f"Se usa porque la referencia solo conoce combinaciones que ya ocurrieron, y el simulador necesita otras: con "
   f"un tope de 10 %, la mayoría de las líneas recortadas son de subcategorías que nunca se vendieron con ese "
   f"descuento. Ninguno de los dos prueba causa: estiman lo que suele pasar en los datos.")

Para simular una línea con descuento mayor que el tope:

- **Precio:** la venta pasa a `Sales × (1 − tope) / (1 − descuento)`. Las unidades no cambian, porque en
  los datos el descuento no se relaciona con la cantidad por línea (correlación 0,01).
- **Margen:** el real más el cambio que predice el modelo al pasar del descuento original al tope.
- **Pedidos perdidos:** cuántos clientes dejarían de comprar sin el descuento no se puede estimar con
  estos datos. Es un supuesto (25 % en el escenario principal) y más abajo se ve cómo cambia el resultado.

## Escenario principal: tope de 20 % en toda la empresa

In [ ]:
p = simular(tope=20, perdida=0.25)
vb, vs, gb, gs = p.v_base.sum(), p.v1.sum(), p.g_base.sum(), p.g1.sum()
kpis([("Ventas 2018", usd(vs), f"base {usd(vb)} · {(vs / vb - 1) * 100:+.1f} %"),
      ("Ganancia 2018", usd(gs), f"base {usd(gb)} · {(gs / gb - 1) * 100:+.1f} %"),
      ("Margen", f"{gs / vs * 100:.1f} %", f"base {gb / vb * 100:.1f} %")])

### Impacto de las recomendaciones en el negocio

In [ ]:
# Lectura del impacto: líneas afectadas por el tope y hasta qué % de pedidos perdidos conviene
e = esc[esc.tope == 20]


def umbral(k):
    """Desde qué % de pedidos perdidos el tope deja de mejorar ventas o ganancia (None: nunca mejora)."""
    antes, despues = e[f"{k}_cambian_antes"].sum(), e[f"{k}_cambian_despues"].sum()
    g0, g1 = despues - antes, -antes  # cambio si no se pierde nada / si se pierden todos esos pedidos
    return None if g0 <= 0 else (1.0 if g1 >= 0 else g0 / (g0 - g1))


ug, uv = umbral("ganancia"), umbral("ventas")
partes = [f"Con un tope de 20 % en toda la empresa se recorta el descuento de {int(e.lineas_cambian.sum()):,} "
          f"líneas de pedido de 2017."]
if ug is None:
    partes.append("Este tope no mejora la ganancia ni aunque no se pierda ningún pedido.")
elif ug >= 1:
    partes.append("La ganancia mejora aunque se pierdan todos esos pedidos, porque hoy dejan pérdida.")
else:
    partes.append(f"La ganancia mejora mientras se pierda menos del {ug * 100:.0f} % de esos pedidos.")
if uv is not None and uv < 1:
    partes.append(f"Las ventas no bajan si se pierde menos del {uv * 100:.0f} %.")
partes.append(f"En total, la ganancia de 2018 pasa de {usd(gb)} a {usd(gs)} ({(gs / gb - 1) * 100:+.0f} %).")
md(" ".join(partes))

In [ ]:
# Real 2015–2017, proyección base y proyección con el tope; las líneas de 2018 arrancan en el último mes real
real = load_mensual()
real = real[real.Mes.dt.year >= 2015].groupby("Mes", as_index=False).Sales.sum()
m = p.groupby("Mes", as_index=False)[["v_base", "v1"]].sum()
x = list(real.Mes.tail(1)) + list(m.Mes)
fig = go.Figure([
    go.Scatter(x=real.Mes, y=real.Sales, name="Real", mode="lines+markers", line=dict(color=REAL, width=2),
               marker_size=4),
    go.Scatter(x=x, y=list(real.Sales.tail(1)) + list(m.v_base), name="Proyección base 2018", mode="lines+markers",
               line=dict(color=PROY, width=2.5, dash="dash"), marker_size=5),
    go.Scatter(x=x, y=list(real.Sales.tail(1)) + list(m.v1), name="Con tope de 20 %", mode="lines+markers",
               line=dict(color=POS, width=3), marker_size=6),
])
fig.update_yaxes(title="Ventas del mes (USD)", tickprefix="$", tickformat=",.0f")
show(style_fig(fig, 380))

Con menos descuento, las líneas recortadas pagan más por las mismas unidades: aunque se pierda una
parte de esos pedidos, las ventas casi no cambian y la ganancia sube, porque los descuentos de más de
20 % hoy dejan pérdida.

## ¿Dónde conviene poner el tope?

Ganancia que habría tenido 2017 con cada tope, según cuántos de los pedidos recortados se pierdan.

In [ ]:
real_2017 = ganancia_2017(80)
fig = go.Figure()
colores = {0: POS, 25: REAL, 50: PROY, 100: NEG}
mejores = {}
for perd, color in colores.items():
    g = [ganancia_2017(t, perdida=perd / 100) for t in TOPES]
    mejores[perd] = (TOPES[int(np.argmax(g))], max(g))
    fig.add_trace(go.Scatter(x=TOPES, y=g, mode="lines+markers", name=f"Se pierde el {perd} % de esos pedidos",
                             line=dict(color=color, width=2.5), marker_size=6,
                             hovertemplate="Tope %{x} %<br>Ganancia %{y:$,.0f}<extra></extra>"))
fig.add_hline(y=real_2017, line_dash="dash", line_color=REF, annotation_text=f"Real 2017: {usd(real_2017)}",
              annotation_position="bottom right")
fig.update_xaxes(title="Tope de descuento (%)", ticksuffix=" %", tickvals=TOPES)
fig.update_yaxes(title="Ganancia 2017 simulada (USD)", tickprefix="$", tickformat=",.0f")
show(style_fig(fig, 400))
md("\n".join(f"- Si se pierde el {k} % de los pedidos recortados, el mejor tope es {t} % ({usd(g)})."
              for k, (t, g) in mejores.items()))

Los topes bajos (0–15 %) dependen mucho del supuesto, porque recortan descuentos pequeños que hoy sí
dejan ganancia. El tope de 20 % es la conclusión más sólida: aun si se perdieran todos los pedidos
recortados, es el mejor tope y deja más ganancia que la real.

## Por región

Ganancia 2018 al aplicar el tope de 20 % en una sola región, con 25 % de pedidos perdidos y en el
peor caso (se pierden todos).

In [ ]:
base = simular().g_base.sum()
filas = []
for reg in ["Central", "East", "South", "West"]:
    g25, g100 = simular(tope=20, region=reg, perdida=0.25).g1.sum(), simular(tope=20, region=reg, perdida=1).g1.sum()
    filas.append({"Región": reg, "Con 25 % de pérdida": g25 - base, "Peor caso": g100 - base})
t = pd.DataFrame(filas).sort_values("Con 25 % de pérdida", ascending=False)
tabla(t).format({"Con 25 % de pérdida": lambda v: f"{usd(v)} más", "Peor caso": lambda v: f"{usd(v)} más"})

En todas las regiones el tope mejora la ganancia incluso en el peor caso. La región donde más aporta es
la que más descuentos altos concentra, coherente con el hallazgo del EDA.

## Las tres recomendaciones juntas

Tope de 20 % (25 % de pedidos perdidos), temporada alta +5 % y Technology +5 %. Los dos últimos efectos
son supuestos ilustrativos: los datos no registran campañas, así que el modelo no puede estimarlos.

In [ ]:
p = simular(tope=20, perdida=0.25, temporada=5, categoria="Technology", impulso=5)
g = p[["g_base", "g1", "g2", "g3"]].sum()
pasos = ["Ganancia base", "Tope de descuento", "Temporada alta +5 %", "Technology +5 %", "Con recomendaciones"]
valores = [g.g_base, g.g1 - g.g_base, g.g2 - g.g1, g.g3 - g.g2, g.g3]
fig = go.Figure(go.Waterfall(
    x=pasos, y=valores, measure=["absolute", "relative", "relative", "relative", "total"],
    text=[usd(v) if k in (0, 4) else f"+{usd(v)}" for k, v in enumerate(valores)], textposition="outside",
    increasing=dict(marker_color=POS), decreasing=dict(marker_color=NEG), totals=dict(marker_color=PROY),
    connector=dict(line=dict(color=REF, width=1, dash="dot"))))
fig.update_yaxes(title="Ganancia 2018 (USD)", tickprefix="$", tickformat=",.0f", range=[0, valores[-1] * 1.15])
style_fig(fig, 380)
fig.update_layout(showlegend=False)
show(fig)
md(f"La ganancia de 2018 pasa de {usd(g.g_base)} a {usd(g.g3)} ({(g.g3 / g.g_base - 1) * 100:+.0f} %) y las "
   f"ventas, de {usd(p.v_base.sum())} a {usd(p.v3.sum())}. El tope aporta {usd(g.g1 - g.g_base)}: casi todo el "
   f"cambio. Vender más con un margen de {g.g1 / p.v1.sum() * 100:.0f} % suma poco a la ganancia.")

## Supuestos y límites

- La proyección base es la de Random Forest (capítulo *Proyección 2018*). La ganancia base de 2018 usa
  el margen de 2017 de cada categoría en cada región.
- Con menos descuento, cada pedido conserva sus unidades y paga más por ellas.
- Cuántos clientes dejarían de comprar sin el descuento es un supuesto; por eso se muestran varios.
- Son escenarios estimados, no causas probadas. Antes de aplicar una recomendación en toda la
  empresa, conviene probarla en una región y compararla con las demás.